# Example 01 - Analytic Double-Well Potential

This notebook mirrors `examples/01_analytic_doublewell.py`:

1. Define a simple analytic 2D double-well potential.
2. Detect basins automatically on the potential grid.
3. Estimate bidirectional MFPTs from overdamped Langevin replicas.
4. Plot the potential landscape and detected basin minima.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "stochkin").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "examples" / "data"
OUT_DIR = ROOT / "notebooks" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Notebook output directory: {OUT_DIR}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import stochkin as sk

from stochkin.plotting import _apply_pub_axes, _apply_pub_cbar
from stochkin.style import publication_style


In [ ]:
KT = 0.5
GAMMA = 1.0
DT = 1e-3
MAX_TIME = 5e4
N_TRIALS = 200


def double_well(x):
    x = np.asarray(x, dtype=float)
    U = (x[0] ** 2 - 1.0) ** 2 + x[1] ** 2
    F = np.array(
        [
            -4.0 * x[0] * (x[0] ** 2 - 1.0),
            -2.0 * x[1],
        ]
    )
    return U, F


In [ ]:
basin_net = sk.build_basin_network_from_potential(
    double_well,
    xlim=(-2.0, 2.0),
    ylim=(-1.5, 1.5),
    nx=101,
    ny=101,
)

print(f"Found {basin_net.n_basins} basins")
for basin in basin_net.basins:
    print(f"Basin {basin.id}: minimum ~ {basin.minimum}")


In [ ]:
basin_a, basin_b = basin_net.basins[0], basin_net.basins[1]

result = sk.compute_bidirectional_mfpt(
    double_well,
    lambda x: basin_net.which_basin(x) == basin_a.id,
    lambda x: basin_net.which_basin(x) == basin_b.id,
    kT=KT,
    gamma=GAMMA,
    n_trials=N_TRIALS,
    max_time=MAX_TIME,
    dt=DT,
    regime="overdamped",
    boundsA=basin_a.bounds,
    boundsB=basin_b.bounds,
    processes=1,
)

mfpt_01 = result["A_to_B"]["mean"]
mfpt_10 = result["B_to_A"]["mean"]
print(f"MFPT 0->1 = {mfpt_01:.2e} tau")
print(f"MFPT 1->0 = {mfpt_10:.2e} tau")


In [ ]:
X = np.linspace(-2.0, 2.0, 200)
Y = np.linspace(-1.5, 1.5, 200)
Xg, Yg = np.meshgrid(X, Y, indexing="ij")
U_grid = np.vectorize(lambda xi, yi: double_well([xi, yi])[0])(Xg, Yg)

output_path = OUT_DIR / "01_doublewell.png"

with publication_style():
    fig, ax = plt.subplots(figsize=(3.3, 2.8))
    cs = ax.contourf(
        X,
        Y,
        U_grid.T / KT,
        levels=np.linspace(0, 10, 20),
        cmap="viridis_r",
    )
    cbar = fig.colorbar(cs, ax=ax)
    _apply_pub_cbar(cbar, label=r"$U / k_\mathrm{B}T$")

    for basin in basin_net.basins:
        mx, my = float(basin.minimum[0]), float(basin.minimum[1])
        ax.plot(mx, my, "o", color="white", ms=8)
        ax.text(mx, my, f" B{basin.id}", color="white", va="center", fontsize=9)

    _apply_pub_axes(ax, r"$x_1$", r"$x_2$", "Double-well potential")
    fig.tight_layout()
    fig.savefig(output_path, dpi=300)

print(f"Saved {output_path.relative_to(ROOT)}")
plt.show()
